In [ ]:
# ARC-AGI-3 :: novelty-guided informed search (no LLM, no internet, no weights)
# BUILD_STAMP attests WHICH CODE ACTUALLY RAN. A push that does not change this hash is a no-op,
# and a "success" message is not evidence the new code executed -- read this back out of the log.
BUILD_STAMP = "4d030d63754a3515"
BUILT_UTC   = "2026-08-29T22:41:45.924730+00:00"
print("BUILD_STAMP", BUILD_STAMP, "BUILT_UTC", BUILT_UTC, flush=True)

import os, sys, time
print("python", sys.version, flush=True)
print("KAGGLE_IS_COMPETITION_RERUN =", os.getenv("KAGGLE_IS_COMPETITION_RERUN"), flush=True)
T0 = time.time()


In [ ]:
!pip install --no-index --find-links=/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv 2>&1 | tail -3


In [ ]:
%%writefile /tmp/my_agent.py
"""MyAgent — novelty-guided informed search for ARC-AGI-3. No LLM, no internet, no weights.

WHY THIS SHAPE [VERIFIED, docs/PRIOR_ART.md]: every result that has actually won on ARC-AGI-3 is
built from two primitives, and the organisers summarise both as "informed search":
    2025 Preview 1st (12.58%) — a model of WHICH ACTIONS CHANGE THE FRAME.
    2025 Preview 2nd  (6.71%) — a DIRECTED STATE GRAPH over observed frames.
Both are cheap and neither needs a model. We implement exactly those and nothing else.

CRASH-SAFETY IS A DELIBERATE FEATURE, NOT DEFENSIVENESS. Competition Mode scores EVERY
environment whether or not we touched it, `make` may be called ONCE per environment with no
retries, and a per-game overrun raises "scorecard not produced in time" which FAILS THE RUN.
So: every decision is wrapped, and any internal failure DEGRADES to a legal random action rather
than propagating. One bad environment must cost ~1.8 points, never the run.

GATEWAY PARITY. On Kaggle the frames come from the gateway sidecar, not the local engine, and
`available_actions` arrives as RAW INTS rather than GameAction members. Every read of the frame
below is written to accept both shapes. This is the single most likely place for a local-green /
Kaggle-red divergence, so it is handled at the SOURCE (`_avail`) and every branch inherits it.
"""

import hashlib
import os
import random
import time
from collections import defaultdict

from arcengine import FrameData, GameAction, GameState

from ..agent import Agent

# Wall clock is a RUN-FAILURE mode, not a score term: overrunning fails the whole run, so both
# a per-game and a global budget are enforced. These are OURS, chosen against the 9 h kernel cap.
GAME_TIME_LIMIT_S = float(os.getenv("GAME_TIME_LIMIT_S", "300"))     # 5 min per environment
GLOBAL_TIME_LIMIT_S = float(os.getenv("GLOBAL_TIME_LIMIT_S", str(7.5 * 3600)))
_PROCESS_START = time.time()


def _grid_bytes(frame):
    """Raw bytes of the frame grid, tolerating list-of-2D-lists or a numpy-ish array."""
    g = getattr(frame, "frame", None)
    if not g:
        return b""
    try:
        out = bytearray()
        for layer in g:
            for row in layer:
                out.extend(bytes(bytearray((int(v) & 0xFF) for v in row)))
        return bytes(out)
    except Exception:
        return repr(g).encode("utf-8", "replace")


class MyAgent(Agent):
    """Count-based novelty search over a learned state graph, with an action-effect prior."""

    # The base class default is 80, which would stop us mid-level on almost every environment.
    # We rely on the ORGANISERS' 5x-human per-level termination and on our own TIME budget
    # instead, because the human baseline is not observable through the gateway.
    MAX_ACTIONS = float("inf")

    OPTIMISTIC_BONUS = 1.6
    DEAD_ACTION_FLOOR = 0.05
    LEVEL_REWARD = 8.0
    EPSILON = 0.10
    EXPLORE_FRACTION = 0.6
    N_COORD_CANDIDATES = 24
    LATTICE_STEP = 8

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.rng = random.Random(0xA3C ^ (hash(self.game_id) & 0xFFFFFFFF))
        self.start_time = time.time()
        self.visits = defaultdict(int)
        self.transitions = {}
        self.effect = defaultdict(lambda: [0, 0])
        self.level_actions = set()
        self._prev_hash = None
        self._prev_key = None
        self._prev_levels = 0
        self._resets = 0

    # ------------------------------------------------------------------ frame readers
    @staticmethod
    def _avail(latest_frame):
        """Available actions as GameAction members. THE gateway-parity seam — see module docstring.

        The gateway sends raw ints; the local engine sends members. Normalising HERE means every
        branch below inherits the fix, rather than each call site being patched and the next one
        added silently unpatched.
        """
        raw = getattr(latest_frame, "available_actions", None) or []
        out = []
        for a in raw:
            try:
                out.append(a if isinstance(a, GameAction) else GameAction.from_id(int(a)))
            except Exception:
                continue
        return [a for a in out if a is not GameAction.RESET]

    def _hash(self, frame):
        return hashlib.blake2b(_grid_bytes(frame), digest_size=16).digest()

    # ------------------------------------------------------------------ the learning half
    def _key(self, action, data=None):
        if action is GameAction.ACTION6 and data:
            return "A6:%d,%d" % (int(data.get("x", 0)) // 8, int(data.get("y", 0)) // 8)
        return action.name

    def _observe(self, latest_frame):
        h = self._hash(latest_frame)
        self.visits[h] += 1
        levels = int(getattr(latest_frame, "levels_completed", 0) or 0)
        if self._prev_hash is not None and self._prev_key is not None:
            k = self._prev_key
            self.transitions[(self._prev_hash, k)] = h
            self.effect[k][0] += int(h != self._prev_hash)
            self.effect[k][1] += 1
            if levels > self._prev_levels:
                self.level_actions.add(k)     # the only true reward the environment emits
        self._prev_levels = levels
        self._prev_hash = h
        self._prev_key = None

    # ------------------------------------------------------------------ the deciding half
    def _coords(self, latest_frame):
        """Rare-colour centroids (likely interactive objects), then a uniform lattice."""
        pts = []
        try:
            g = getattr(latest_frame, "frame", None)
            if g:
                grid = g[0]
                counts = defaultdict(int)
                pos = defaultdict(list)
                for y, row in enumerate(grid):
                    for x, v in enumerate(row):
                        v = int(v)
                        counts[v] += 1
                        if len(pos[v]) < 64:
                            pos[v].append((x, y))
                if counts:
                    bg = max(counts, key=lambda k: counts[k])
                    for v in sorted(counts, key=lambda k: counts[k]):
                        if v == bg or not pos[v]:
                            continue
                        xs = [p[0] for p in pos[v]]
                        ys = [p[1] for p in pos[v]]
                        pts.append((sum(xs) // len(xs), sum(ys) // len(ys)))
                        if len(pts) >= self.N_COORD_CANDIDATES // 2:
                            break
        except Exception:
            pts = []
        s = self.LATTICE_STEP
        lat = [(x, y) for x in range(s // 2, 64, s) for y in range(s // 2, 64, s)]
        self.rng.shuffle(lat)
        pts.extend(lat[: max(0, self.N_COORD_CANDIDATES - len(pts))])
        return pts[: self.N_COORD_CANDIDATES]

    def _effect_prior(self, k):
        ch, tr = self.effect[k]
        return 1.0 if tr == 0 else max(ch / tr, self.DEAD_ACTION_FLOOR)

    def is_done(self, frames, latest_frame):
        if latest_frame.state is GameState.WIN:
            return True
        now = time.time()
        if (now - self.start_time) >= GAME_TIME_LIMIT_S:
            return True
        if (now - _PROCESS_START) >= GLOBAL_TIME_LIMIT_S:
            return True
        return False

    def choose_action(self, frames, latest_frame):
        """Never raises. Any internal failure degrades to a legal random action."""
        try:
            return self._choose(frames, latest_frame)
        except BaseException:
            import traceback
            print("!! choose_action failed, degrading to random", flush=True)
            traceback.print_exc()
            return self._fallback(latest_frame)

    def _fallback(self, latest_frame):
        try:
            av = self._avail(latest_frame) or [a for a in GameAction if a is not GameAction.RESET]
        except Exception:
            av = [GameAction.ACTION1]
        a = self.rng.choice(av)
        if a.is_complex():
            a.set_data({"x": self.rng.randrange(64), "y": self.rng.randrange(64)})
        return a

    def _choose(self, frames, latest_frame):
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            # A GAME_OVER is recoverable: the engine downgrades a game reset to a LEVEL reset, and
            # abandoning here forfeits every remaining level -- which carry the HIGHEST weight.
            self._resets += 1
            self._prev_hash, self._prev_key = None, None
            return GameAction.RESET

        self._observe(latest_frame)
        avail = self._avail(latest_frame)
        if not avail:
            return GameAction.RESET

        # Anneal novelty -> exploitation across the game's TIME budget. Exploration and
        # exploitation come out of one budget; that trade-off is the central design constraint.
        prog = min((time.time() - self.start_time) / max(GAME_TIME_LIMIT_S, 1e-6), 1.0)
        anneal = max(0.0, 1.0 - prog / self.EXPLORE_FRACTION)

        h = self._hash(latest_frame)
        cands = []
        for a in avail:
            if a is GameAction.ACTION6:
                for (x, y) in self._coords(latest_frame):
                    d = {"x": x, "y": y}
                    cands.append((a, d, self._key(a, d)))
            else:
                cands.append((a, None, self._key(a)))
        if not cands:
            return self._fallback(latest_frame)

        best, best_s = [], -1e18
        for (a, d, k) in cands:
            nxt = self.transitions.get((h, k))
            if nxt is None:
                nov = self.OPTIMISTIC_BONUS
            else:
                nov = 1.0 / (1.0 + self.visits[nxt]) ** 0.5
                if nxt == h:
                    nov *= 0.15          # a self-loop is a wasted action
            s = self._effect_prior(k) * (nov * anneal + (1.0 - anneal))
            if k in self.level_actions:
                s *= self.LEVEL_REWARD
            if s > best_s:
                best_s, best = s, [(a, d, k)]
            elif s == best_s:
                best.append((a, d, k))

        # An epsilon floor: the model is learned from few samples and CAN be confidently wrong,
        # and a deterministic argmax over a wrong model is an absorbing trap.
        a, d, k = self.rng.choice(cands) if self.rng.random() < self.EPSILON else self.rng.choice(best)
        self._prev_key = k
        if a.is_complex():
            a.set_data(d or {"x": self.rng.randrange(64), "y": self.rng.randrange(64)})
        return a


In [ ]:
import os, sys, time, traceback

# ☠ DEGRADE, NEVER ABORT. Kaggle DISCARDS an errored rerun together with any output file, so a
# raise here costs the slot AND every trace of what happened. Exit 0 deliberately; print the
# traceback under a loud banner so a degraded run is still greppable evidence.
def _main():
    if not os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        print("== Phase A (commit): validating execution, not behaviour ==", flush=True)
        import pandas as pd
        pd.DataFrame(data=[['1_0', '1', True, 1]],
                     columns=['row_id', 'game_id', 'end_of_game', 'score']
                     ).to_parquet('/kaggle/working/submission.parquet', index=False)
        print("wrote dummy submission.parquet", flush=True)
        return

    print("== Phase B (competition rerun) ==", flush=True)
    COMP = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3"
    os.system("curl --fail --retry 999 --retry-all-errors --retry-delay 5 "
              "--retry-max-time 600 http://gateway:8001/api/games")
    os.system(f"cp -r {COMP}/ARC-AGI-3-Agents /kaggle/working/ARC-AGI-3-Agents")
    os.system("cp /tmp/my_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py")

    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write(
            "from typing import Type, cast\n"
            "from dotenv import load_dotenv\n"
            "from .agent import Agent, Playback\n"
            "from .swarm import Swarm\n"
            "from .templates.random_agent import Random\n"
            "from .templates.my_agent import MyAgent\n"
            "load_dotenv()\n"
            "AVAILABLE_AGENTS: dict[str, Type[Agent]] = {\n"
            '    "random": Random,\n'
            '    "myagent": MyAgent,\n'
            "}\n")

    # OPERATION_MODE=online is what the OFFICIAL sample uses. The gateway is in competition mode
    # server-side; setting "competition" here ourselves is an untested deviation and this is a
    # one-shot scored run, so we match the sample exactly.
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("SCHEME=http\nHOST=gateway\nPORT=8001\n"
                "ARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\n"
                "OPERATION_MODE=online\nENVIRONMENTS_DIR=\n"
                "RECORDINGS_DIR=/kaggle/working/server_recording\n")

    rc = os.system("cd /kaggle/working/ARC-AGI-3-Agents && MPLBACKEND=agg "
                   "GAME_TIME_LIMIT_S=300 python main.py --agent myagent")
    print("main.py rc =", rc, flush=True)

try:
    _main()
except BaseException:
    print("!! " * 24, flush=True)
    print("!! AGENT RUN FAILED -- DEGRADING SO THE RUN IS RETAINED AS EVIDENCE", flush=True)
    traceback.print_exc()
    print("!! " * 24, flush=True)
    try:
        import pandas as pd
        p = '/kaggle/working/submission.parquet'
        if not os.path.exists(p):   # never overwrite a real result with the fallback
            pd.DataFrame(data=[['1_0', '1', True, 1]],
                         columns=['row_id','game_id','end_of_game','score']
                         ).to_parquet(p, index=False)
            print("wrote fallback submission.parquet", flush=True)
    except BaseException:
        traceback.print_exc()

print("TOTAL ELAPSED", round(time.time() - T0, 1), "s", flush=True)
sys.exit(0)
